## XBRL US API - Python example  

### Authenticate for access token 
Run the cell below, then type your XBRL US Web account email, account password, Client ID, and secret as noted, pressing the Enter key on the keyboard after each entry.

In [ ]:
print('Enter your XBRL US Web account email: ')
import os, re, sys, json
import requests
import pandas as pd
import numpy as np
import getpass
email = input()
print('Account password: ')
password = getpass.getpass()
print('Client ID: ')
clientid = getpass.getpass()
print('Client secret: ')
secret = getpass.getpass()

body_auth = {'username' : ''.join(email), 
            'client_id': ''.join(clientid), 
            'client_secret' : ''.join(secret), 
            'password' : ''.join(password), 
            'grant_type' : 'password', 
            'platform' : 'ipynb' }
endpoint_auth = 'https://api.xbrl.us/oauth2/token'
res = requests.post(endpoint_auth, data=body_auth)
auth_json = res.json()

if 'error' in auth_json:
    print ("\n\nThere was a problem generating an access token with these credentials. Run the first cell again to enter credentials.")
else:
    print ("\n\nYour access token expires in 60 minutes. After it expires, run the cell immediately below this one to generate a new token and continue to use the query cell. \n\nFor now, skip ahead to the section 'Make a Query'.")
access_token = auth_json['access_token']
refresh_token = auth_json['refresh_token']
newaccess = ''
newrefresh = ''
#print('access token: ' + access_token + ' refresh token: ' + refresh_token)


#### Refresh token 
The cell below is only needed to refresh an expired access token after 60 minutes. When the access token no longer returns results, run the cell below to refresh the access token or re-enter credentials by running the cell above. Until the refresh token process is needed, skip ahead to **Make a Query**. 


In [ ]:
token = token if newrefresh != '' else refresh_token 

refresh_auth = {'client_id': ''.join(clientid), 
            'client_secret' : ''.join(secret), 
            'grant_type' : 'refresh_token', 
            'platform' : 'ipynb', 
            'refresh_token' : ''.join(token) }
refreshres = requests.post(endpoint_auth, data=refresh_auth)
refresh_json = refreshres.json()
access_token = refresh_json['access_token']
refresh_token = refresh_json['refresh_token']#print('access token: ' + access_token + 'refresh token: ' + refresh_token)
print('Your access token is refreshed for 60 minutes. If it expires again, run this cell to generate a new token and continue to use the query cells below.')
print(access_token)

### Make a query 
After the access token confirmation appears above, you can modify the query below and use the **_Cell >> Run All Below_** menu option with the cell **immediately below this text** to run the query for updated results. 
  
Refer to XBRL API documentation at https://xbrlus.github.io/xbrl-api/#/Facts/getFactDetails for other endpoints and parameters to filter and return. 
  
Non-members might not be able to return all data for a query - consider joining XBRL US for comprehensive access - https://xbrl.us/join.

In [ ]:
# Define the parameters for the filter and fields to be returned, run the loop to return results

queried_count = 0
offset_value = 0
res_df = []

# Define the parameters of the query

XBRL_Elements = ['EffectiveIncomeTaxRateReconciliationAtFederalStatutoryIncomeTaxRate',
'EffectiveIncomeTaxRateReconciliationTaxCutsAndJobsActOf2017Percent',
'EffectiveIncomeTaxRateReconciliationTaxCutsAndJobsActOf2017TransitionTaxOnAccumulatedForeignEarningsPercent',
'EffectiveIncomeTaxRateReconciliationStateAndLocalIncomeTaxes',
'EffectiveIncomeTaxRateReconciliationForeignIncomeTaxRateDifferential',
'EffectiveIncomeTaxRateReconciliationTaxCredits',
'EffectiveIncomeTaxRateReconciliationChangeInEnactedTaxRate',
'EffectiveIncomeTaxRateReconciliationChangeInDeferredTaxAssetsValuationAllowance',
'EffectiveIncomeTaxRateReconciliationShareBasedCompensationExcessTaxBenefitPercent',
'EffectiveIncomeTaxRateReconciliationOtherAdjustments',
'EffectiveIncomeTaxRateReconciliationOtherReconcilingItemsPercent',
'EffectiveIncomeTaxRateContinuingOperations',
'IncomeTaxReconciliationIncomeTaxExpenseBenefitAtFederalStatutoryIncomeTaxRate',
'EffectiveIncomeTaxRateReconciliationTaxCutsAndJobsActOf2017Amount',
'EffectiveIncomeTaxRateReconciliationTaxCutsAndJobsActOf2017TransitionTaxOnAccumulatedForeignEarningsAmount',
'IncomeTaxReconciliationStateAndLocalIncomeTaxes',
'IncomeTaxReconciliationForeignIncomeTaxRateDifferential',
'IncomeTaxReconciliationTaxCredits',
'IncomeTaxReconciliationOtherReconcilingItems'
                ]
sic_code = ['2834'
         ]

years = ['2020'
         ]

periods = ['Y']

# Define data fields to return (multi-sort based on order)

fields = ['period.fiscal-year.sort(DESC)',
         'entity.name.sort(ASC)',
         'concept.local-name.sort(ASC)',
         'fact.value',
         'unit',
         'fact.decimals',
         'report.filing-date'
         ]

params = {
         'concept.local-name': ','.join(XBRL_Elements),
         'report.sic-code': ','.join(sic_code),
         'period.fiscal-year': ','.join(years),
         'period.fiscal-period': ','.join(periods),
         'fields': ','.join(fields),
         'fact.ultimus': 'TRUE'
         #'fact.offset': 0
         }

# Execute the query with loop for all results

search_endpoint = 'https://api.xbrl.us/api/v1/fact/search'
orig_fields = params['fields']

count = 0
while True:
    res = requests.get(search_endpoint, params=params, headers={'Authorization' : 'Bearer {}'.format(access_token)})
    res_json = res.json()
    if 'error' in res_json:
        print('There was an error: {}'.format(res_json['error_description']))
        break

    res_df += res_json['data']

    if res_json['paging']['count'] < res_json['paging']['limit']:
        break
    else:
        params['fields'] = orig_fields + ',fact.offset({})'.format(res_json['paging']['limit'])

if not 'error' in res_json:
    print("Requested Fields:", orig_fields, "\n\nResponse:\n\n", pd.DataFrame(res_df))  

In [13]:
# Save the output to a file on your computer (modify D:\\results.csv to your system)

df = pd.DataFrame(res_df)
df.to_csv("D:\\results.csv",sep=",")